# 02 Predictive Track

Track A treats SparseTap as next-bit prediction. We run statistical lag scans, Logistic Regression with L1, XGBoost or fallback boosting, and a lightweight MLP. Every method saves artifacts and candidate rule hints.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

SEARCH_ROOTS = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(path for path in SEARCH_ROOTS if (path / "DAY2_data.txt").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sparsetap_config import DEFAULT_CONFIG
from sparsetap_utils import (
    build_supervised_dataset,
    candidate_table,
    prepare_environment,
    run_logistic_l1,
    run_mlp,
    run_pair_scan,
    run_single_lag_scan,
    run_xgboost,
    save_candidates,
    save_table,
)

config = DEFAULT_CONFIG
data, prefix = prepare_environment(config)
dataset = build_supervised_dataset(data, max_lag=config.max_lag)

single = run_single_lag_scan(data, prefix, config)
pair = run_pair_scan(data, prefix, [cand["taps"][0] for cand in single["candidates"][:12]], config)
logistic = run_logistic_l1(dataset, data, prefix, config)
xgb = run_xgboost(dataset, data, prefix, config)
mlp = run_mlp(dataset, data, prefix, config)

predictive_candidates = single["candidates"] + pair["candidates"] + logistic + xgb + mlp
save_candidates(predictive_candidates, config.candidate_dir / "predictive_candidates.json")
predictive_table = candidate_table(predictive_candidates).sort_values(
    ["prefix consistent?", "log-likelihood", "accuracy", "number of taps"],
    ascending=[False, False, False, True],
)
save_table(predictive_table, config.metrics_dir / "predictive_summary.csv")
predictive_table.head(30)


This notebook also supports failure analysis. Predictive methods store notes in metadata so the final report can summarize which baselines were weak, which were only useful as priors, and why raw predictive accuracy alone is not enough.